In [ ]:
from datetime import timedelta  # Imports timedelta to easily shift, add, or subtract specific intervals of calendar time

import numpy as np  # Imports NumPy to run optimized multi-dimensional array operations and statistical calculations
import pandas as pd  # Imports Pandas to structure and manipulate tabular timeseries data inside dataframes
from sklearn.ensemble import RandomForestRegressor  # Imports the Random Forest regressor for tree-based ensemble learning models
from sklearn.linear_model import LinearRegression  # Imports the classical OLS Linear Regression algorithm for trend fitting
from sklearn.metrics import mean_absolute_error, mean_squared_error  # Imports standard loss functions to compute model performance deviations
from sklearn.preprocessing import MinMaxScaler  # Imports scaling utilities to normalize continuous values between zero and one
from statsmodels.tsa.arima.model import ARIMA  # Imports the ARIMA model framework for classical autoregressive timeseries forecasting


def demo_frame(rows=420):  # Defines a helper function to construct mock synthetic financial market data frames for localized testing
    rng = np.random.default_rng(7)  # Seeds a numpy pseudo-random number generator to guarantee consistent matrix mutations across test runs
    dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=rows, freq="B")  # Generates an absolute continuous string of past business days up to the current normalized date
    trend = np.linspace(112, 164, rows)  # Generates an evenly distributed upward vector mimicking structural market asset gains over time
    cycle = 5 * np.sin(np.arange(rows) / 17)  # Introduces a smooth repeating sine wave mutation pattern to replicate asset seasonality waves
    close = trend + cycle + rng.normal(0, 2.1, rows)  # Merges structural trends, cyclical mutations, and Gaussian random noise to generate closing data
    close = np.maximum(close, 5)  # Imposes a hard floor limit ensuring closing prices never clip beneath five credits
    return pd.DataFrame({"Date": dates, "Close": close, "Volume": rng.integers(2_000_000, 9_000_000, rows)})  # Bundles columns into a single dataframe containing random trade volumes


def load_market_frame(symbol, period):  # Defines a data acquisition handler to fetch active public financial metrics from external sources
    try:  # Enters a try-except layer block to safely handle potential network connection drops or module omissions
        import yfinance as yf  # Imports Yahoo Finance on demand to avoid unnecessary performance overhead during bootstrap processes
        frame = yf.download(symbol, period=period, auto_adjust=True, progress=False)  # Downloads live asset histories based on tickers and ranges without progress outputs
        if frame.empty:  # Evaluates whether the downloaded dataset returned any actual records from remote servers
            raise RuntimeError(f"No market data was returned for {symbol}.")  # Throws an exception alerting that the target asset symbol is missing or incorrect
        frame = frame.reset_index()  # Standardizes DataFrame layout maps by shifting date structures from row indexes to ordinary data rows
        if isinstance(frame.columns, pd.MultiIndex):  # Detects if multi-level column indexing was introduced by bulk yfinance operations
            frame.columns = [column[0] for column in frame.columns]  # flattens multi-layered tracking titles down to simple zero-level single strings
        return frame  # Passes the clean, single-indexed timeseries matrix back to calling application layers
    except ImportError as exc:  # Intercepts missing module environments if yfinance is not available on the local machine
        raise RuntimeError("Live market data requires yfinance to be installed.") from exc  # Raises a runtime error instruction defining remediation dependencies


def _normalise(frame):  # Declares an internal layout sanitizer to standardize inbound multi-source file table dimensions
    columns = {str(column).strip().lower(): column for column in frame.columns}  # Assembles case-insensitive cross-reference name dictionaries stripping padding spaces
    date_column = columns.get("date") or columns.get("datetime")  # Identifies suitable chronological indexing structures looking for specific naming variations
    close_column = columns.get("close") or columns.get("adj close")  # Matches suitable continuous targeting series utilizing explicit closing headers
    if not close_column:  # Validates whether a suitable targeting evaluation column was successfully mapped
        raise KeyError("The file must include a Close or Adj Close column.")  # Hard aborts operations highlighting structural format constraints
    if not date_column:  # Handles instances where data streams fail to output explicit timing structures
        dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=len(frame), freq="B")  # Dynamically auto-fabricates matching business date sequences backwards from today
    else:  # Proceeds with explicit calendar conversion rules when native headers match requirements
        dates = pd.to_datetime(frame[date_column], errors="coerce")  # Normalizes time records while mapping bad rows to null tags
    values = pd.to_numeric(frame[close_column], errors="coerce")  # Forces values down into clean float spaces while removing broken characters
    clean = pd.DataFrame({"date": dates, "close": values}).dropna().sort_values("date").drop_duplicates("date")  # Aggregates variables, purges null cells, orders rows chronologically, and clears duplicate entries
    if len(clean) < 80:  # Enforces a critical minimum length threshold requirement for historical data tracking
        raise ValueError("At least 80 valid price observations are required for analysis.")  # Terminates processing routines due to statistical model constraints
    return clean.reset_index(drop=True)  # Returns dataframes built on top of cleanly sequenced linear array keys


def _metrics(actual, predicted):  # Declares an validation processor generating standard evaluation error statistics
    actual, predicted = np.asarray(actual), np.asarray(predicted)  # Enforces strict mapping of input variables to optimized numpy datatypes
    nonzero = np.where(actual == 0, 1, actual)  # Overwrites zeroes to ones on division scales to avoid zero-division infrastructure faults
    return {"mae": round(float(mean_absolute_error(actual, predicted)), 3), "rmse": round(float(np.sqrt(mean_squared_error(actual, predicted))), 3), "mape": round(float(np.mean(np.abs((actual - predicted) / nonzero)) * 100), 2)}  # Returns rounded error metrics calculations


def _supervised(values, window=20):  # Splits flat timeseries lines into supervised machine learning input arrays
    features, target = [], []  # Declares empty memory lists to host input parameter structures and target vectors
    for index in range(window, len(values)):  # Shifts iteratively forward over the collection from the termination point of the first lookback block
        features.append(values[index - window:index])  # Extracts sliced arrays of past observations to represent model feature spaces
        target.append(values[index])  # Pinpoints the immediate trailing price element as the target label to be predicted
    return np.array(features), np.array(target)  # Transforms memory lists back into structured high-performance numpy arrays


def _linear_forecast(values, horizon):  # Fits ordinary least squares regression matrices to generate forward linear trajectories
    x = np.arange(len(values)).reshape(-1, 1)  # Establishes a basic step sequence indexing space acting as the independent variable array
    model = LinearRegression().fit(x, values)  # Calculates line weights matching historical patterns
    future = np.arange(len(values), len(values) + horizon).reshape(-1, 1)  # Projects step index markers into tomorrow's chronological horizon limits
    return model, model.predict(future)  # Extrapolates out future prediction metrics alongside structural object handles


def _random_forest_forecast(values, horizon):  # Trains randomized ensemble regression trees to predict future trends recursively
    window = 20  # Mandates a structural lag constraint lookback window defining historical feature scopes
    features, target = _supervised(values, window)  # Reshapes historical value blocks into analytical regression training matrices
    model = RandomForestRegressor(n_estimators=160, max_depth=8, random_state=42, n_jobs=-1).fit(features, target)  # Trains a random forest ensemble utilizing optimized core execution resources
    history = list(values)  # Converts the array into a dynamic list structure to support recursive appending actions
    forecast = []  # Declares an empty tracker array to register sequential model tracking outputs
    for _ in range(horizon):  # Iterates down sequential time steps to manually step through horizon bounds
        next_value = float(model.predict([history[-window:]])[0])  # Feeds the latest lookback window block to extract a single new scalar estimate
        forecast.append(next_value)  # Registers the freshly calculated step output directly inside our prediction log arrays
        history.append(next_value)  # appends the predicted scalar back into the training log to enable autoregressive lookbacks
    return model, np.array(forecast)  # Passes the ensemble object together with the compiled forward estimation arrays back to the pipeline


def _arima_forecast(values, horizon):  # Leverages statistical auto-regressive moving average engines to track parameters
    model = ARIMA(values, order=(2, 1, 2)).fit()  # Configures and optimizes a standard seasonal ARIMA state matrix across tracking arrays
    return model, np.asarray(model.forecast(horizon))  # Computes and returns structural projections matching explicit horizon counts


